In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 2.2 Build a Regularized Logistic Regression Model
## Model Cycle Step 1: **Build**
- Create pipelines with L2, L1, and ElasticNet regularization
- Same preprocessing as Course 2
- Same data, different regularization settings

## The Model Cycle
### **1. Build** ← We are here
### 2. Train
### 3. Predict
### 4. Evaluate
### 5. Improve

## Setup: Load Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import pickle

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder

pd.options.display.max_columns = None

In [ ]:
data_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
models_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'

df_training = pd.read_csv(f'{data_filepath}training.csv')

print(f"Training data shape: {df_training.shape}")
print(f"\nTarget distribution:")
print(df_training['SEM_3_STATUS'].value_counts(normalize=True))

In [ ]:
X_train = df_training
y_train = df_training['SEM_3_STATUS']

## Quick Recap: Preprocessing from Course 2
- **MinMaxScaler** → bounded values (GPA, rates)
- **StandardScaler** → larger numeric values (units)
- **OneHotEncoder** → categorical columns

In [ ]:
minmax_columns = [
    'HS_GPA',
    'GPA_1', 'GPA_2',
    'DFW_RATE_1', 'DFW_RATE_2'
]

standard_columns = [
    'UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2',
]

categorical_columns = [
    'GENDER',
    'RACE_ETHNICITY',
    'FIRST_GEN_STATUS',
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('minmax', MinMaxScaler(), minmax_columns),
        ('standard', StandardScaler(), standard_columns),
        ('onehot', OneHotEncoder(handle_unknown='ignore', 
                                  drop=['Female', 'Other', 'Unknown'], 
                                  sparse_output=False), categorical_columns)
    ],
    remainder='drop'
)

print("Preprocessor configured successfully.")

## Key Hyperparameters
| Parameter | Description | Values |
|:----------|:------------|:-------|
| `penalty` | Type of regularization | `'l1'`, `'l2'`, `'elasticnet'` |
| `C` | Inverse of regularization strength | float > 0 (default=1.0) |
| `solver` | Optimization algorithm | `'saga'` needed for L1/ElasticNet |
| `l1_ratio` | ElasticNet mixing (1=L1, 0=L2) | float 0 to 1 |

## Build Model 1: L2 (Ridge)
- Shrinks all coefficients proportionally
- Never zeros any out
- Uses `solver='lbfgs'` (default)

In [ ]:
l2_logistic_model = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(
        penalty='l2',
        C=1.0,
        class_weight='balanced',
        solver='lbfgs',
        max_iter=1000,
        random_state=42
    ))
])

print("L2 (Ridge) Logistic Regression Model:")
l2_logistic_model

## Build Model 2: L1 (Lasso)
- Can shrink coefficients to **exactly zero**
- Automatic feature selection
- Requires `solver='saga'`

In [ ]:
l1_logistic_model = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(
        penalty='l1',
        C=1.0,
        class_weight='balanced',
        solver='saga',
        max_iter=1000,
        random_state=42
    ))
])

print("L1 (Lasso) Logistic Regression Model:")
l1_logistic_model

## Build Model 3: ElasticNet
- Combines L1 + L2 penalties
- `l1_ratio=0.5` → equal mix
- Requires `solver='saga'`

In [ ]:
elasticnet_logistic_model = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(
        penalty='elasticnet',
        C=1.0,
        l1_ratio=0.5,
        class_weight='balanced',
        solver='saga',
        max_iter=1000,
        random_state=42
    ))
])

print("ElasticNet Logistic Regression Model:")
elasticnet_logistic_model

## Compare What We Built

In [ ]:
models = {
    'L2 (Ridge)': l2_logistic_model,
    'L1 (Lasso)': l1_logistic_model,
    'ElasticNet': elasticnet_logistic_model
}

print("Models Built:")
print("="*50)
for name, model in models.items():
    classifier = model.named_steps['classifier']
    print(f"\n{name}:")
    print(f"  - penalty: {classifier.penalty}")
    print(f"  - C (regularization): {classifier.C}")
    print(f"  - solver: {classifier.solver}")
    if hasattr(classifier, 'l1_ratio') and classifier.l1_ratio is not None:
        print(f"  - l1_ratio: {classifier.l1_ratio}")

## Save Models for Next Notebook

In [ ]:
import os
os.makedirs(models_filepath, exist_ok=True)

for name, model in models.items():
    filename = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    filepath = f'{models_filepath}{filename}_logistic_model.pkl'
    pickle.dump(model, open(filepath, 'wb'))
    print(f"Saved: {filepath}")

## Key Takeaways
| Model | Characteristics | Best For |
|:------|:---------------|:---------|
| **L2 (Ridge)** | Shrinks all coefficients | Many small effects |
| **L1 (Lasso)** | Zeros out coefficients | Feature selection |
| **ElasticNet** | Combines L1 + L2 | Correlated features |

- All use `class_weight='balanced'` for class imbalance
- All use default `C=1.0` — we'll tune later
- **Next:** 2.3 Train and Compare Models